# SAML-D case-level — freeze, weryfikacja, podłogi

Kontekst: `samld_case_proto.ipynb` dał PASS (pary strukturalne 2-way, ProtoNet 0,64 → 0,83). Ten notebook zamraża przypadki jako dataset `samld-case` i liczy podłogi do kampanii.

**Ujęcie zadania (zapisane w `schema.json`):** przypisanie typologii do już wykrytego przypadku. Granice przypadków wyznacza `Is_laundering`, więc to nie jest wykrywanie.

**Foldy:** bloki po 4 typologie (test 4 / walidacja 4 / reszta trening), zadanie 4-way. Jeśli liczba typologii nie dzieli się przez 4, reszta dostaje dodatkowy fold, dopełniony pierwszymi typologiami z tej samej seedowanej permutacji. Każda typologia jest testowana co najmniej raz.

**STOP (ustalony przed freeze):** jeśli największy przypadek obejmuje > 10% wszystkich podejrzanych transakcji, łączenie po 7 dniach skleja osobne przypadki. Wtedy nie idziemy dalej.

Kernel: `fsl (py3.10)`. Wymaga podmienionych plików: `scripts/bootstrap_fsl_data.py`, `src/fsl/data/samld.py`, `src/fsl/data/registry.py`.

In [ ]:
from fsl.config import TrainConfig
from fsl.data.registry import get_loaders

cfg = TrainConfig(project="dark-data-discovery", bucket="dark-data-discovery-fsl-data",
                  dataset="samld-case", seed=0, fold=0, n_way=4, k_shot=5, query=5, log_to_vertex=False)
load_frozen, _ = get_loaders(cfg)
b, sha = load_frozen(cfg)
W, s = b["windows"], b["schema"]
print(len(W), "przypadkow |", W[s["group_col"]].nunique(), "grup kont | sha", sha[:12])
print(W[s["label_col"]].value_counts().to_string())

In [ ]:
# KOMORKA 1 — freeze samld-case do gs://<bucket>/raw/samld-case/  (kilka minut: CSV + przypadki)
import os, subprocess, sys
from pathlib import Path
HOME = Path("/home/jupyter")
REPO = HOME / "mlops-meta-learning/fsl-projekt"
PROJECT, BUCKET, DS = "dark-data-discovery", "dark-data-discovery-fsl-data", "samld-case"
env = {**os.environ, "SAMLD_CSV": str(HOME / "data_cache/saml_d_source/SAML-D.csv")}
r = subprocess.run([sys.executable, "scripts/bootstrap_fsl_data.py", "--dataset", DS,
                    "--project", PROJECT, "--bucket", BUCKET, "--location", "us-central1",
                    "--force"],                 # nadpisuje poprzedni freeze (nowy SHA)
                   cwd=REPO, env=env, capture_output=True, text=True)
print(r.stdout[-2000:])
print(r.stderr[-6000:])          # logi bootstrapu ida na stderr
assert r.returncode == 0, "freeze nie przeszedl - wklej output"

In [ ]:
# KOMORKA 2 — weryfikacja: ladowanie przez TEN SAM loader co pipeline, STOP na sklejone przypadki
import json, numpy as np, torch
from fsl.config import TrainConfig
from fsl.data.registry import get_loaders

cfg = TrainConfig(project=PROJECT, bucket=BUCKET, dataset=DS, seed=0, fold=0,
                  n_way=4, k_shot=5, query=5, log_to_vertex=False)
load_frozen, build = get_loaders(cfg)
bundle, sha = load_frozen(cfg)
sch = bundle["schema"]
big = sch["largest_case"]
print(f"sha {sha[:12]} | przypadkow {len(bundle['windows']):,} | cech {len(sch['feature_names'])} | foldow {sch['n_folds']}")
print(f"najwiekszy przypadek: {big['n_tx']} transakcji = {big['share_of_flagged']:.1%} podejrzanych")
if big["share_of_flagged"] > 0.10:
    raise RuntimeError("STOP: najwiekszy przypadek > 10% podejrzanych - laczenie skleja przypadki")
print("odrzucone typologie:", sch["dropped_classes"] or "-",
      "| dodatkowy fold dla:", sch.get("leftover_tested_in_extra_fold") or "-")
for f in bundle["folds"]:
    print(f"fold {f['fold']}: test {f['test']} | val {f['val']}")
untested = sorted(set(sch["classes"]) - {c for f in bundle["folds"] for c in f["test"]})
assert not untested, f"typologie nigdy nietestowane: {untested}"
for k in range(sch["n_folds"]):                   # kazdy fold musi dac epizod train/val/test
    c = TrainConfig(project=PROJECT, bucket=BUCKET, dataset=DS, seed=0, fold=k,
                    n_way=4, k_shot=5, query=5, log_to_vertex=False)
    for split, smp in zip(("train", "val", "test"), build(bundle, c)):
        x, y = smp.sample()
        assert x.shape == (4 * 10, len(sch["feature_names"])), (split, x.shape)
print("wszystkie foldy probkuja poprawnie")

In [ ]:
# KOMORKA 3 — podlogi: losowy (nietrenowany) MLP na tescie, 3 ziarna x 400 epizodow.
# Regula progu ustalona PRZED policzeniem podlog: +0.10 albo usuniecie polowy bledow podlogi,
# co mniejsze. Dla podlogi <= 0.80 to dokladnie stara regula (floor + 0.10); wyzej chroni
# przed progiem >= 1.0, ktorego zaden model nie przejdzie.
from fsl.models.mlp import MLP
from fsl.training.loop import evaluate

FLOORS = {}
for k in range(sch["n_folds"]):
    accs = []
    for seed in (0, 1, 2):
        c = TrainConfig(project=PROJECT, bucket=BUCKET, dataset=DS, seed=seed, fold=k,
                        n_way=4, k_shot=5, query=5, log_to_vertex=False)
        _, _, te = build(bundle, c)
        torch.manual_seed(seed)
        m = MLP(in_features=len(sch["feature_names"]), hid=64)
        accs.append(float(evaluate(m, te, c, 400, torch.device("cpu")).mean()))
    FLOORS[k] = round(float(np.mean(accs)), 4)
THRESHOLDS = {k: round(min(f + 0.10, f + 0.5 * (1 - f)), 4) for k, f in FLOORS.items()}
for k in FLOORS:
    print(f"fold {k}: podloga {FLOORS[k]:.4f} -> prog {THRESHOLDS[k]:.4f}")
print("\nFLOORS =", FLOORS)
(HOME / "data_cache/samld_case_floors.json").write_text(
    json.dumps({"sha": sha, "floors": FLOORS, "thresholds": THRESHOLDS}, indent=2))